In [ ]:
#!/usr/bin/env python
"""
Embedding Pipeline

Turns raw text (the LLM's generated utterance, the ground-truth tutor
utterance, and the dialogue context) into numerical sentence embeddings,
which every downstream analysis (MAI, clustering, similarity scoring)
operates on instead of the raw text directly.

Input:  one or more llm_<tag>.csv files (from 03_generate_llm_responses.py)
Output: per-file .npy arrays + a metadata CSV, saved under embeddings/

Fields encoded:
- context               the dialogue so far (shared across conditions)
- LLM_generation        the model's generated tutor utterance
- next_tutor_utterance  the ground-truth tutor utterance
- next_tutor_utterance_n1 / _n2  supplementary adaptivity-analysis targets, if present

Run once per model's output file, before running the main analysis script
(05_run_all_figures_and_tables.py).
"""

import os

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# ── Config -------------------------------------------------------------------
# FILE_TAGS / INPUT_FILES together tell the script which llm_<tag>.csv files
# to process. Add one entry per model you want to embed.
FILE_TAGS = ["llm_mistrallatest"]
INPUT_FILES = {
    "llm_mistrallatest": "/data/saga/llm_mistrallatest.csv.gz",
}

EMBED_MODEL = "all-mpnet-base-v2"  # sentence-transformers model used throughout the paper
BATCH_SIZE = 64
OUTPUT_DIR = "embeddings"


def encode_and_save(texts, label: str, file_tag: str, model: SentenceTransformer,
                     output_dir: str = OUTPUT_DIR):
    """
    Encode one list of strings into embeddings and save to disk as a .npy
    array. Handles missing values (NaN) by encoding an empty string, then
    zeroing out that row's embedding afterward, so a missing generation
    doesn't accidentally get treated as similar to anything.
    """
    nan_mask = pd.isna(texts)
    texts_clean = ["" if pd.isna(t) else str(t) for t in texts]
    print(f"  Encoding {label} ({len(texts_clean)} texts, {nan_mask.sum()} NaN rows)...")

    # normalize_embeddings=True means every embedding has unit length,
    # which is what lets later cosine-similarity calculations reduce to
    # a simple dot product (see the MAI methodology in the main analysis).
    embeddings = model.encode(
        texts_clean,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    if nan_mask.sum() > 0:
        embeddings[nan_mask] = 0.0  # zero out rows that had no real text to begin with

    out_path = os.path.join(output_dir, f"{label}__{file_tag}.npy")
    np.save(out_path, embeddings)
    print(f"  Saved -> {out_path}  shape={embeddings.shape}")
    return embeddings


def process_file(file_tag: str, input_path: str, model: SentenceTransformer):
    """
    Load one model's generation CSV and encode all of its relevant text
    fields. Also saves a small metadata CSV alongside the embeddings, so
    later analysis scripts can match embedding array rows back to which
    model/condition/segment they came from without re-loading the (much
    larger) full generation file.
    """
    print(f"\n{'=' * 60}\nProcessing: {file_tag}\n{'=' * 60}")

    df = pd.read_csv(input_path)
    # Some generation runs accidentally wrote a duplicate header row into
    # the middle of the CSV (from resumed runs) -- filter those out.
    if "experiment" in df.columns:
        df = df[df["experiment"] != "experiment"].reset_index(drop=True)

    print(f"  Rows: {len(df)}")
    print(f"  Experiments: {df['experiment'].value_counts().to_dict()}")
    print(f"  Conditions:  {df['context_condition'].value_counts().to_dict()}")
    print(f"  Models:      {df['LLM_version'].unique().tolist()}")

    # n+1 / n+2 columns are only used for supplementary adaptivity analyses,
    # so they're optional -- only encode them if they're actually present.
    has_n1 = "next_tutor_utterance_n1" in df.columns
    has_n2 = "next_tutor_utterance_n2" in df.columns

    encode_and_save(df["context"].tolist(), "context_emb", file_tag, model)
    encode_and_save(df["LLM_generation"].tolist(), "generation_emb", file_tag, model)
    encode_and_save(df["next_tutor_utterance"].tolist(), "tutor_emb", file_tag, model)
    if has_n1:
        encode_and_save(df["next_tutor_utterance_n1"].tolist(), "tutor_n1_emb", file_tag, model)
    if has_n2:
        encode_and_save(df["next_tutor_utterance_n2"].tolist(), "tutor_n2_emb", file_tag, model)

    # Save just the small identifying columns (not the full text fields --
    # those are already captured in the embeddings themselves) so later
    # scripts can join embedding rows back to model/segment/condition info.
    meta_cols = [c for c in ["observation_id", "LLM_version", "experiment", "context_condition",
                              "transcriptID", "segment_id", "timestamp", "hyperparameters"]
                 if c in df.columns]
    meta_path = os.path.join(OUTPUT_DIR, f"metadata__{file_tag}.csv")
    df[meta_cols].to_csv(meta_path, index=True, index_label="row_idx")
    print(f"  Metadata -> {meta_path}")


def sanity_check():
    """
    After processing, quickly re-load each saved embedding array and check
    that (a) it exists, (b) its shape makes sense, and (c) normalization
    actually worked (unit-length vectors have norm == 1, so "non-zero rows"
    and the norm range below should both look sane -- catches silent
    encoding failures before they propagate into the main analysis).
    """
    for file_tag in FILE_TAGS:
        print(f"\n{file_tag}")
        for label in ["context_emb", "generation_emb", "tutor_emb", "tutor_n1_emb", "tutor_n2_emb"]:
            path = os.path.join(OUTPUT_DIR, f"{label}__{file_tag}.npy")
            if not os.path.exists(path):
                print(f"  {label}: NOT FOUND")
                continue
            arr = np.load(path)
            norms = np.linalg.norm(arr, axis=1)
            nonzero = (norms > 0).sum()
            print(f"  {label}: shape={arr.shape}  non-zero rows={nonzero}/{len(arr)}  "
                  f"norm range=[{norms[norms > 0].min():.3f}, {norms[norms > 0].max():.3f}]")


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"Output directory: {OUTPUT_DIR}")

    model = SentenceTransformer(EMBED_MODEL)
    print(f"Loaded model: {EMBED_MODEL}")

    for file_tag, input_path in INPUT_FILES.items():
        process_file(file_tag, input_path, model)

    print("\nAll files processed.")
    sanity_check()


if __name__ == "__main__":
    main()